In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

plt.style.use("default")
plt.rcParams.update({
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

PALETTE = {
    "primary": "#2E86AB",
    "secondary": "#A23B72",
    "accent": "#F18F01",
    "good": "#06A77D",
    "bad": "#D62828",
    "neutral": "#6C757D",
}

def wrap_labels(labels, width=18):
    import textwrap
    return ["\n".join(textwrap.wrap(str(x), width=width)) for x in labels]

def fmt_eur(x):
    if pd.isna(x): return "NA"
    x = float(x)
    if abs(x) >= 1_000_000: return f"€{x/1_000_000:.1f}M"
    if abs(x) >= 1_000: return f"€{x/1_000:.1f}k"
    return f"€{x:,.0f}"

def read_table(full_name: str) -> pd.DataFrame:
    return spark.table(full_name).toPandas()

# ----------------------------
# Load tables (created by SQL)
# ----------------------------
overall = read_table("production.supply_analytics.pricing_feature_audit_overall")
decomp  = read_table("production.supply_analytics.pricing_feature_decomposition")
countp  = read_table("production.supply_analytics.pricing_feature_count_performance")
cooc    = read_table("production.supply_analytics.pricing_feature_cooccurrence")
sup     = read_table("production.supply_analytics.supplier_pricing_profile")
segsum  = read_table("production.supply_analytics.segment_pricing_summary")
segfeat = read_table("production.supply_analytics.pricing_features_by_segment_rich")
model   = read_table("production.supply_analytics.pricing_model_performance")
system  = read_table("production.supply_analytics.pricing_system_performance")
combo   = read_table("production.supply_analytics.pricing_feature_combinations")

# ----------------------------
# Executive KPIs (supplier-side)
# ----------------------------
kpi = {
    "Tours (perf)": int(segsum["tours_perf"].fillna(0).sum()),
    "Suppliers (perf)": int(segsum["suppliers_perf"].fillna(0).sum()),
    "Total GMV (perf, €M)": float(segsum["total_gmv_millions"].fillna(0).sum()),
}
fig, ax = plt.subplots(figsize=(12, 2.2))
ax.axis("off")
txt = "   ".join([f"{k}: {v:,.0f}" if "€M" not in k else f"{k}: {v:,.1f}" for k, v in kpi.items()])
ax.text(0.01, 0.5, txt, fontsize=14, fontweight="bold", va="center")
plt.tight_layout()
plt.show()

# ----------------------------
# 1) Adoption audit (universe)
# ----------------------------
ad = overall.sort_values("adoption_universe_pct", ascending=True)
fig, ax = plt.subplots(figsize=(10, 0.6*len(ad)+2))
ax.barh(ad["feature_name"], ad["adoption_universe_pct"], color=PALETTE["primary"], edgecolor="black")
ax.set_title("Adoption by feature (universe)")
ax.set_xlabel("Adoption (% tours)")
ax.set_xlim(0, 100)
for i, (pct, n) in enumerate(zip(ad["adoption_universe_pct"], ad["tours_universe_with"])):
    ax.text(pct + 1, i, f"{pct:.1f}% ({int(n):,} tours)", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()

# ----------------------------
# 2) Impact audit (median lifts) with sample guardrails
# ----------------------------
MIN_WITH = 200
MIN_WITHOUT = 200
impact = overall[(overall["tours_perf_with"] >= MIN_WITH) & (overall["tours_perf_without"] >= MIN_WITHOUT)].copy()

def lift_bar(df, value_col, title, color):
    d = df.sort_values(value_col, ascending=True)
    fig, ax = plt.subplots(figsize=(10, 0.6*len(d)+2))
    ax.barh(d["feature_name"], d[value_col], color=color, edgecolor="black")
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Lift (%)")
    for i, (v, n1, n0) in enumerate(zip(d[value_col], d["tours_perf_with"], d["tours_perf_without"])):
        ax.text(v + (1 if v >= 0 else -1), i,
                f"{v:.1f}% (with {int(n1):,}, without {int(n0):,})",
                va="center", ha="left" if v >= 0 else "right", fontsize=9)
    plt.tight_layout()
    plt.show()

lift_bar(impact, "gmv_lift_pct_median", "GMV lift (median) with vs without feature", PALETTE["good"])
lift_bar(impact, "bookings_lift_pct_median", "Bookings lift (median) with vs without feature", PALETTE["accent"])
lift_bar(impact, "aov_lift_pct_median", "AOV lift (median) with vs without feature", PALETTE["secondary"])

# Prioritization scatter (adoption vs lift)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(impact["adoption_universe_pct"], impact["gmv_lift_pct_median"], s=220, alpha=0.85,
           color=PALETTE["secondary"], edgecolor="black")
for _, r in impact.iterrows():
    ax.annotate(r["feature_name"], (r["adoption_universe_pct"], r["gmv_lift_pct_median"]),
                xytext=(6, 6), textcoords="offset points", fontsize=9)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Prioritization view: low adoption + high GMV lift")
ax.set_xlabel("Adoption (% tours, universe)")
ax.set_ylabel("GMV lift (%), median (perf population)")
plt.tight_layout()
plt.show()

# ----------------------------
# 3) GMV decomposition: bookings vs AOV contribution
# ----------------------------
d = decomp.replace([np.inf, -np.inf], np.nan).dropna(subset=["log_gmv_lift", "log_bookings_lift", "log_aov_lift"])
d = d.sort_values("gmv_lift_pct_median", ascending=False)

fig, ax = plt.subplots(figsize=(12, 0.55*len(d)+2))
y = np.arange(len(d))
ax.barh(y, d["log_bookings_lift"], color=PALETTE["accent"], edgecolor="black", label="Bookings contribution (log)")
ax.barh(y, d["log_aov_lift"], left=d["log_bookings_lift"], color=PALETTE["secondary"], edgecolor="black", label="AOV contribution (log)")
ax.set_yticks(y)
ax.set_yticklabels(d["feature_name"])
ax.set_title("GMV lift decomposition (approx): bookings + AOV contributions")
ax.set_xlabel("Log-lift contributions (additive approximation)")
ax.legend(loc="lower right")
for i, gmv_lift in enumerate(d["gmv_lift_pct_median"]):
    ax.text(d["log_bookings_lift"].iloc[i] + d["log_aov_lift"].iloc[i], i,
            f"  GMV {gmv_lift:.1f}%", va="center", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.show()

# ----------------------------
# 4) Feature-count ladder (native-only): outcomes vs number of controllable features
# ----------------------------
d = countp.sort_values("pricing_feature_count")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d["pricing_feature_count"], d["median_gmv"], marker="o", linewidth=2, color=PALETTE["primary"])
ax.set_title("Median GMV by number of pricing-controllable features (native only)")
ax.set_xlabel("Pricing feature count (individual, group, addons, scale)")
ax.set_ylabel("Median GMV per tour")
ax.set_yscale("log")
for x, yv, n in zip(d["pricing_feature_count"], d["median_gmv"], d["tours"]):
    ax.text(x, yv, f"  n={int(n):,}", fontsize=8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(d["pricing_feature_count"], d["median_bookings"], marker="o", linewidth=2, color=PALETTE["accent"])
ax.set_title("Median bookings by number of pricing-controllable features (native only)")
ax.set_xlabel("Pricing feature count")
ax.set_ylabel("Median bookings per tour")
ax.set_yscale("log")
for x, yv, n in zip(d["pricing_feature_count"], d["median_bookings"], d["tours"]):
    ax.text(x, yv, f"  n={int(n):,}", fontsize=8)
plt.tight_layout()
plt.show()

# ----------------------------
# 5) Supplier concentration + supplier size bands
# ----------------------------
s = sup[["supplier_id", "gmv_total"]].copy().sort_values("gmv_total", ascending=False)
s["cum_gmv_share"] = s["gmv_total"].cumsum() / s["gmv_total"].sum()
s["cum_suppliers"] = (np.arange(len(s)) + 1) / len(s)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(s["cum_suppliers"], s["cum_gmv_share"], color=PALETTE["primary"], linewidth=2)
ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1)
ax.set_title("GMV concentration across suppliers (Lorenz curve)")
ax.set_xlabel("Cumulative share of suppliers")
ax.set_ylabel("Cumulative share of GMV")
plt.tight_layout()
plt.show()

band_order = ["1", "2-3", "4-7", "8-15", "16+"]
band = (sup.groupby("supplier_size_band")
          .agg(
              suppliers=("supplier_id", "count"),
              tours=("tours", "sum"),
              gmv_total=("gmv_total", "sum"),
              avg_feature_count=("avg_feature_count", "mean"),
              median_gmv_per_tour=("median_gmv_per_tour", "median"),
              median_bookings_per_tour=("median_bookings_per_tour", "median"),
          )
          .reset_index())
band["supplier_size_band"] = pd.Categorical(band["supplier_size_band"], categories=band_order, ordered=True)
band = band.sort_values("supplier_size_band")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(band["supplier_size_band"], band["avg_feature_count"], color=PALETTE["neutral"], edgecolor="black")
ax.set_title("Supplier size vs adoption intensity (avg # pricing features)")
ax.set_xlabel("Supplier size band (# tours)")
ax.set_ylabel("Avg pricing feature count")
for i, (v, n) in enumerate(zip(band["avg_feature_count"], band["suppliers"])):
    ax.text(i, v, f"  n={int(n):,}", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

# ----------------------------
# 6) Feature co-occurrence (native-only) + performance when both present
# ----------------------------
pv = cooc.pivot(index="f1", columns="f2", values="both_pct_of_tours")
fig, ax = plt.subplots(figsize=(10, 5))
mat = pv.values
im = ax.imshow(mat, aspect="auto", cmap="Blues", vmin=np.nanmin(mat), vmax=np.nanmax(mat))
ax.set_xticks(np.arange(pv.shape[1])); ax.set_xticklabels(wrap_labels(pv.columns, 14))
ax.set_yticks(np.arange(pv.shape[0])); ax.set_yticklabels(wrap_labels(pv.index, 14))
ax.set_title("Feature co-occurrence prevalence (native): % tours with both")
for i in range(pv.shape[0]):
    for j in range(pv.shape[1]):
        val = mat[i, j]
        ax.text(j, i, "NA" if np.isnan(val) else f"{val:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, label="% tours with both")
plt.tight_layout()
plt.show()

c2 = cooc.copy()
c2["delta_both_vs_neither"] = c2["median_gmv_both"] - c2["median_gmv_neither"]
c2 = c2.sort_values("delta_both_vs_neither", ascending=False).head(10)
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh((c2["f1"] + " + " + c2["f2"]).iloc[::-1], c2["delta_both_vs_neither"].iloc[::-1],
        color=PALETTE["good"], edgecolor="black")
ax.set_title("Top feature pairs: median GMV (both) - median GMV (neither)")
ax.set_xlabel("€ difference (median GMV)")
plt.tight_layout()
plt.show()

# ----------------------------
# 7) Segment heterogeneity (top segments by GMV)
# ----------------------------
TOP_SEGMENTS = 10
top_segs = segsum.sort_values("total_gmv_millions", ascending=False).head(TOP_SEGMENTS)["segment"].tolist()
sf = segfeat[segfeat["segment"].isin(top_segs)].copy()

def heatmap(df_long, value_col, title, vmin=None, vmax=None, cmap="YlGn"):
    p = df_long.pivot(index="segment", columns="feature_name", values=value_col).loc[top_segs]
    mat = p.values
    rows, cols = p.index.tolist(), p.columns.tolist()

    if vmin is None: vmin = np.nanpercentile(mat, 5)
    if vmax is None: vmax = np.nanpercentile(mat, 95)

    fig, ax = plt.subplots(figsize=(1.15*len(cols)+4, 0.6*len(rows)+2))
    im = ax.imshow(mat, aspect="auto", vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(np.arange(len(cols))); ax.set_xticklabels(wrap_labels(cols, 12))
    ax.set_yticks(np.arange(len(rows))); ax.set_yticklabels(wrap_labels(rows, 18))
    ax.set_title(title)

    for i in range(len(rows)):
        for j in range(len(cols)):
            val = mat[i, j]
            txt = "NA" if np.isnan(val) else f"{val:.0f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=9,
                    color="white" if (not np.isnan(val) and abs(val) > 0.7*max(abs(vmin), abs(vmax))) else "black",
                    fontweight="bold")

    plt.colorbar(im, ax=ax, label=value_col)
    plt.tight_layout()
    plt.show()

heatmap(sf, "adoption_pct", "Adoption by segment × feature (top segments by GMV)", vmin=0, vmax=100, cmap="YlGn")
heatmap(sf, "gmv_lift_pct_median", "Median GMV lift by segment × feature (native)", vmin=-50, vmax=150, cmap="RdYlGn")
heatmap(sf, "bookings_lift_pct_median", "Median bookings lift by segment × feature (native)", vmin=-50, vmax=150, cmap="RdYlGn")
heatmap(sf, "aov_lift_pct_median", "Median AOV lift by segment × feature (native)", vmin=-50, vmax=150, cmap="RdYlGn")

# ----------------------------
# 8) Combos audit
# ----------------------------
c = combo[combo["tours"] >= 100].copy()

top_perf = c.sort_values("median_gmv", ascending=False).head(12).sort_values("median_gmv", ascending=True)
fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_perf["feature_combo"], top_perf["median_gmv"], color=PALETTE["primary"], edgecolor="black")
ax.set_title("Top combos by median GMV (native only)")
ax.set_xlabel("Median GMV")
ax.set_xscale("log")
for i, (v, n) in enumerate(zip(top_perf["median_gmv"], top_perf["tours"])):
    ax.text(v, i, f"  {fmt_eur(v)} ({int(n):,} tours)", va="center", fontsize=9)
plt.tight_layout()
plt.show()

top_lift = c.sort_values("gmv_lift_vs_baseline_pct", ascending=False).head(15).sort_values("gmv_lift_vs_baseline_pct")
fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_lift["feature_combo"], top_lift["gmv_lift_vs_baseline_pct"], color=PALETTE["accent"], edgecolor="black")
ax.axvline(0, color="black", linewidth=1)
ax.set_title("GMV lift vs baseline (Individual Only) by combo")
ax.set_xlabel("Lift (%)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(c["tours"], c["avg_gmv"], s=np.sqrt(c["total_gmv_millions"].clip(lower=0))*60,
           alpha=0.7, color=PALETTE["secondary"], edgecolor="black")
for _, r in c.nlargest(10, "total_gmv_millions").iterrows():
    ax.annotate(r["feature_combo"], (r["tours"], r["avg_gmv"]), xytext=(6,6), textcoords="offset points", fontsize=8)
ax.set_title("Combos: prevalence vs performance (bubble size = total GMV)")
ax.set_xlabel("Tours in combo")
ax.set_ylabel("Avg GMV per tour")
ax.set_xscale("log"); ax.set_yscale("log")
plt.tight_layout()
plt.show()

# ----------------------------
# 9) Action table (ranking proxy, not causal): segment scale × adoption gap × positive lift
# ----------------------------
seg_scale = segsum[["segment", "total_gmv_millions"]].copy()
x = sf.merge(seg_scale, on="segment", how="left")

x["adoption_gap"] = 1 - x["adoption_pct"] / 100.0
x["lift_pos"] = x["gmv_lift_pct_median"].clip(lower=0) / 100.0
x["opportunity_score"] = x["total_gmv_millions"].fillna(0) * x["adoption_gap"].fillna(0) * x["lift_pos"].fillna(0)

action = x.sort_values("opportunity_score", ascending=False)[
    ["segment", "feature_name", "adoption_pct", "gmv_lift_pct_median",
     "bookings_lift_pct_median", "aov_lift_pct_median",
     "tours_with", "tours_without", "total_gmv_millions", "opportunity_score"]
].head(30)

action
